# Retail Sales Analytics

This notebook inspects the cleaned Sample Superstore data, validates the business fields, and creates exploratory charts. It reads only local project files and does not invent missing values or results.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'superstore_cleaned.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['Order Date', 'Ship Date'])
sns.set_theme(style='whitegrid', palette='colorblind')
df.head()

In [ ]:
print(f'Rows: {len(df):,}; columns: {len(df.columns)}')
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().to_frame('missing_values'))
print('Duplicate rows:', df.duplicated().sum())
print('Potential high-sales threshold (99th percentile):', df['Sales'].quantile(0.99))

## Cleaning decisions

The script removes exact duplicate rows and rows missing required analytical fields. It keeps valid high-value transactions and reports the 99th-percentile sales threshold as an outlier review aid rather than deleting those records.

In [ ]:
monthly = (df.assign(Month=df['Order Date'].dt.to_period('M').dt.to_timestamp())
             .groupby('Month', as_index=False)
             .agg(Revenue=('Sales', 'sum'), Profit=('Profit', 'sum')))
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
sns.lineplot(data=monthly, x='Month', y='Revenue', marker='o', ax=axes[0])
sns.lineplot(data=monthly, x='Month', y='Profit', marker='o', color='darkorange', ax=axes[1])
axes[0].set(title='Monthly Revenue', ylabel='Revenue')
axes[1].set(title='Monthly Profit', ylabel='Profit', xlabel='Month')
plt.tight_layout()

In [ ]:
category = df.groupby('Category', as_index=False).agg(Revenue=('Sales', 'sum'), Profit=('Profit', 'sum')).sort_values('Revenue', ascending=False)
region = df.groupby('Region', as_index=False)['Sales'].sum().rename(columns={'Sales': 'Revenue'}).sort_values('Revenue', ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=category, x='Revenue', y='Category', ax=axes[0])
sns.barplot(data=region, x='Revenue', y='Region', ax=axes[1])
axes[0].set(title='Revenue by Category', xlabel='Revenue')
axes[1].set(title='Revenue by Region', xlabel='Revenue')
plt.tight_layout()

In [ ]:
discount = df.groupby('Discount', as_index=False).agg(Profit=('Profit', 'sum'), Revenue=('Sales', 'sum'))
sns.scatterplot(data=discount, x='Discount', y='Profit', size='Revenue', sizes=(30, 300), legend=False)
plt.title('Discount versus Profit')
plt.xlabel('Discount rate')
plt.ylabel('Total profit')
plt.show()
print('This chart shows association in the observed data; it does not prove that discount changes caused profit changes.')